In [1]:
# Universal Recursive Tuning (URT) Framework - Plasma Control Demo for Google Colab
# Based on the Enhanced URT v2.0 Framework (October 2025)
# This notebook demonstrates plasma stabilization using URT variants.
# Run cells sequentially. No additional installs needed (uses pre-installed libs).

# Cell 1: Imports and Base URT Class
import numpy as np
import torch
import torch.nn as nn
import time
from typing import Dict, List, Optional, Union, Callable, Tuple
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import scipy.special
from scipy import stats
import warnings
import sys

# Base URT Class (from manuscript Section 2.1)
class UniversalRecursiveTuning:
    """Base URT framework with global stability guarantees and Lyapunov analysis"""

    def __init__(self, alpha: float = 1.155, theta_h: float = 2.4,
                 beta: float = 0.235, state_dim: int = 100,
                 device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.alpha = alpha
        self.theta_h = theta_h
        self.beta = beta
        self.state_dim = state_dim
        self.device = device
        self.convergence_history = []
        self.lyapunov_history = []

        # Verify initial stability
        self.verify_stability()

    def verify_stability(self):
        """Verify global contraction condition with enhanced checks"""
        kappa = self.beta * self.alpha * (1 + self.theta_h)
        if kappa >= 1.0:
            raise ValueError(f"System unstable: κ={kappa:.3f} >= 1")

        # Additional stability margin check
        stability_margin = 1.0 - kappa
        if stability_margin < 0.01:
            warnings.warn(f"Low stability margin: {stability_margin:.3f}")

        print(f"Stability verified: κ={kappa:.3f}, margin: {stability_margin:.3f}")

    def phi(self, P: Union[np.ndarray, torch.Tensor]) -> Union[np.ndarray, torch.Tensor]:
        """Base nonlinearity function with smooth gradients"""
        if isinstance(P, torch.Tensor):
            return torch.where(torch.abs(P) <= torch.pi,
                             torch.sin(P),
                             torch.sign(P))
        else:
            return np.where(np.abs(P) <= np.pi,
                          np.sin(P),
                          np.sign(P))

    def construct_lyapunov_functional(self, P: Union[np.ndarray, torch.Tensor],
                                    P_next: Union[np.ndarray, torch.Tensor]) -> Dict:
        """Construct and analyze Lyapunov functional V(P) = PᵀP"""
        if isinstance(P, torch.Tensor):
            V = torch.norm(P)**2
            V_next = torch.norm(P_next)**2
        else:
            V = np.linalg.norm(P)**2
            V_next = np.linalg.norm(P_next)**2

        delta_V = V_next - V
        kappa = self.beta * self.alpha * (1 + self.theta_h)
        theoretical_bound = (kappa**2 - 1) * V

        lyapunov_data = {
            'V_current': float(V),
            'V_next': float(V_next),
            'delta_V': float(delta_V),
            'theoretical_bound': float(theoretical_bound),
            'lyapunov_decrease_verified': bool(delta_V <= theoretical_bound),
            'contraction_rate': float(kappa)
        }

        self.lyapunov_history.append(lyapunov_data)
        return lyapunov_data

    def step(self, P: Union[np.ndarray, torch.Tensor],
             u_input: float = 0.05) -> Union[np.ndarray, torch.Tensor]:
        """Core URT update step with Lyapunov monitoring"""
        phi_P = self.phi(P)
        nonlinear_term = self.alpha * (P - self.theta_h * phi_P)

        if isinstance(P, torch.Tensor):
            P_next = self.beta * (nonlinear_term + u_input * torch.ones_like(P))
        else:
            P_next = self.beta * (nonlinear_term + u_input * np.ones_like(P))

        # Lyapunov analysis
        self.construct_lyapunov_functional(P, P_next)

        # Record convergence
        error = torch.norm(P_next) if isinstance(P_next, torch.Tensor) else np.linalg.norm(P_next)
        self.convergence_history.append({
            'step': len(self.convergence_history),
            'error': float(error),
            'kappa': float(self.beta * self.alpha * (1 + self.theta_h))
        })

        return P_next

    def simulate(self, P0: Union[np.ndarray, torch.Tensor],
                 steps: int = 100, u_input: float = 0.05) -> List:
        """Complete simulation run with comprehensive monitoring"""
        trajectory = [P0.copy() if isinstance(P0, np.ndarray) else P0.clone()]
        P = P0

        for i in range(steps):
            P = self.step(P, u_input)
            trajectory.append(P.copy() if isinstance(P, np.ndarray) else P.clone())

        return trajectory

    def get_lyapunov_summary(self) -> Dict:
        """Generate Lyapunov stability summary"""
        if not self.lyapunov_history:
            return {}

        decreases = [entry['lyapunov_decrease_verified'] for entry in self.lyapunov_history]
        success_rate = np.mean(decreases)

        return {
            'lyapunov_success_rate': success_rate,
            'total_steps': len(self.lyapunov_history),
            'average_contraction': np.mean([entry['contraction_rate'] for entry in self.lyapunov_history]),
            'worst_lyapunov_change': np.min([entry['delta_V'] for entry in self.lyapunov_history])
        }

# Cell 2: Adaptive URT Class (Section 2.2 - Simplified for Demo)
class AdaptiveURT(UniversalRecursiveTuning):
    """URT with adaptive β-tuning for performance optimization"""

    def __init__(self, alpha: float = 1.155, theta_h: float = 2.4,
                 beta_min: float = 0.235, beta_max: float = 0.5,
                 state_dim: int = 100, confidence_level: float = 0.95):
        super().__init__(alpha, theta_h, (beta_min + beta_max)/2, state_dim)
        self.beta_min = beta_min
        self.beta_max = beta_max
        self.P_prev = None
        self.confidence_level = confidence_level
        self.performance_metrics = {
            'contraction_rates': [],
            'adaptive_betas': [],
            'stability_margins': [],
            'performance_scores': []
        }

    def adaptive_beta(self, P: np.ndarray, P_prev: np.ndarray) -> float:
        """Adapt β based on local convergence rate with enhanced stability"""
        if P_prev is None:
            return self.beta_min

        current_error = np.linalg.norm(P)
        prev_error = np.linalg.norm(P_prev)

        if prev_error == 0:
            return self.beta_min

        local_contraction = current_error / prev_error

        # Enhanced adaptation with hysteresis
        if local_contraction < 0.7:  # Excellent convergence
            beta = self.beta_max * 1.1  # Slight aggression
        elif local_contraction < 0.85:  # Good convergence
            beta = self.beta_max
        elif local_contraction > 0.98:  # Poor convergence
            beta = self.beta_min * 0.9  # Extra conservatism
        else:
            # Smooth interpolation with cubic smoothing
            t = (local_contraction - 0.7) / (0.98 - 0.7)
            t = np.clip(t, 0, 1)
            # Cubic smoothing for smoother transitions
            t_smooth = 3*t**2 - 2*t**3
            beta = self.beta_max * (1 - t_smooth) + self.beta_min * t_smooth

        # Enhanced global stability constraint with margin
        kappa = beta * self.alpha * (1 + self.theta_h)
        stability_margin = 0.95  # 5% safety margin
        if kappa >= stability_margin:
            beta = (stability_margin - 0.01) / (self.alpha * (1 + self.theta_h))

        # Record metrics
        self.performance_metrics['contraction_rates'].append(local_contraction)
        self.performance_metrics['adaptive_betas'].append(beta)
        self.performance_metrics['stability_margins'].append(stability_margin - kappa)

        # Performance scoring
        performance_score = (1 - local_contraction) * (beta / self.beta_max)
        self.performance_metrics['performance_scores'].append(performance_score)

        return beta

    def step(self, P: np.ndarray, u_input: float = 0.05) -> np.ndarray:
        """Enhanced adaptive step with performance optimization"""
        P_prev = P.copy() if self.P_prev is None else self.P_prev

        current_beta = self.adaptive_beta(P, P_prev)
        self.beta = current_beta  # Update for convergence tracking

        nonlinear_term = self.alpha * (P - self.theta_h * self.phi(P))
        P_next = current_beta * (nonlinear_term + u_input * np.ones_like(P))

        self.P_prev = P.copy()
        return P_next

# Cell 3: Plasma Simulation Wrapper (Custom for Fusion Plasma Demo)
class PlasmaSimulator:
    """Simplified plasma dynamics simulator with URT control for fusion stabilization"""

    def __init__(self, state_dim: int = 100, urt_controller=None):
        self.state_dim = state_dim
        self.urt = urt_controller or AdaptiveURT(state_dim=state_dim)
        self.plasma_state = np.random.normal(0, 1.0, state_dim)  # Initial perturbed plasma state
        self.time_steps = []
        self.state_norms = []
        self.control_inputs = []
        self.mode_suppression = []  # Track resonant mode suppression

    def plasma_dynamics(self, state: np.ndarray, control: np.ndarray, dt: float = 0.01) -> np.ndarray:
        """Simplified nonlinear plasma dynamics: MHD-like with resonant modes"""
        # Base plasma evolution (simplified Lorentz-like for demo)
        dx = np.zeros_like(state)
        for i in range(len(state)):
            # Nonlinear coupling
            coupling = np.sum(np.sin(state[(i-1)%len(state)] * state[(i+1)%len(state)]))
            dx[i] = -0.1 * state[i] + 0.05 * coupling + 0.1 * np.random.normal(0, 0.01)  # Noise

        # Add resonant mode (e.g., kink mode simulation)
        resonant_mode = 0.2 * np.sin(2 * np.pi * np.arange(len(state)) / 10 + time.time())  # Time-varying
        dx += resonant_mode * 0.15  # Perturbation strength

        # Apply control
        dx += control * dt

        return state + dx * dt

    def simulate_plasma_with_control(self, steps: int = 200, u_base: float = 0.05):
        """Run plasma simulation with URT control, tracking mode suppression"""
        trajectory = [self.plasma_state.copy()]
        current_state = self.plasma_state.copy()

        for step in range(steps):
            # Compute control via URT
            control = self.urt.step(current_state, u_base)
            self.control_inputs.append(np.mean(control))

            # Evolve plasma
            current_state = self.plasma_dynamics(current_state, control)
            trajectory.append(current_state.copy())

            # Track norm and suppression (inverse of resonant component)
            norm = np.linalg.norm(current_state)
            self.state_norms.append(norm)
            self.time_steps.append(step * 0.01)  # dt=0.01s

            # Mode suppression metric (correlation with resonant mode)
            resonant_comp = np.dot(current_state, np.sin(2 * np.pi * np.arange(len(current_state)) / 10))
            suppression = 1 - abs(resonant_comp) / (norm + 1e-6)
            self.mode_suppression.append(suppression)

        self.plasma_state = current_state  # Final state
        return trajectory

    def plot_results(self):
        """Plot plasma stabilization results"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle('Plasma Stabilization with URT Control')

        # State norm convergence
        axes[0, 0].plot(self.time_steps, self.state_norms)
        axes[0, 0].set_title('Plasma State Norm (Convergence)')
        axes[0, 0].set_xlabel('Time (s)')
        axes[0, 0].set_ylabel('||State||')
        axes[0, 0].grid(True)

        # Mode suppression
        axes[0, 1].plot(self.time_steps, self.mode_suppression)
        axes[0, 1].set_title('Resonant Mode Suppression (%)')
        axes[0, 1].set_xlabel('Time (s)')
        axes[0, 1].set_ylabel('Suppression')
        axes[0, 1].grid(True)

        # Control input
        axes[1, 0].plot(self.time_steps, self.control_inputs)
        axes[1, 0].set_title('Applied Control Input')
        axes[1, 0].set_xlabel('Time (s)')
        axes[1, 0].set_ylabel('u (mean)')
        axes[1, 0].grid(True)

        # Lyapunov summary if available
        if self.urt.lyapunov_history:
            lyap_success = np.mean([e['lyapunov_decrease_verified'] for e in self.urt.lyapunov_history[-len(self.time_steps):]])
            axes[1, 1].bar(['Lyapunov Success Rate'], [lyap_success])
            axes[1, 1].set_title('Stability Verification')
            axes[1, 1].set_ylabel('Rate')

        plt.tight_layout()
        plt.show()

        # Print summary
        final_suppression = np.mean(self.mode_suppression[-50:]) * 100
        print(f"Average Final Mode Suppression: {final_suppression:.1f}%")
        print(f"Final State Norm: {self.state_norms[-1]:.4f}")
        print(f"Lyapunov Success Rate: {self.urt.get_lyapunov_summary().get('lyapunov_success_rate', 0):.3f}")

# Cell 4: Run Plasma Demo
print("Initializing Plasma Simulator with Adaptive URT...")
plasma_sim = PlasmaSimulator(state_dim=50)  # Smaller dim for faster Colab run

print("Running plasma stabilization simulation...")
trajectory = plasma_sim.simulate_plasma_with_control(steps=150)

print("Plotting results...")
plasma_sim.plot_results()

# Cell 5: Benchmark Summary (Quick Stats)
print("\n--- Quick Benchmark ---")
print(f"Convergence Steps to <0.1 norm: {next((i for i, n in enumerate(plasma_sim.state_norms) if n < 0.1), len(plasma_sim.state_norms))}")
print(f"Peak Suppression: {max(plasma_sim.mode_suppression)*100:.1f}%")
print(f"Control Effort (mean |u|): {np.mean(np.abs(plasma_sim.control_inputs)):.4f}")

# Optional: Compare with Base URT
base_urt = UniversalRecursiveTuning(state_dim=50)
base_sim = PlasmaSimulator(state_dim=50, urt_controller=base_urt)
base_trajectory = base_sim.simulate_plasma_with_control(steps=150)
base_suppression = np.mean(base_sim.mode_suppression[-50:]) * 100
adaptive_suppression = np.mean(plasma_sim.mode_suppression[-50:]) * 100
print(f"\nComparison: Adaptive URT Suppression: {adaptive_suppression:.1f}% vs Base: {base_suppression:.1f}% (Improvement: {adaptive_suppression - base_suppression:.1f}%)")

# Cell 6: Enhanced Hybrid URT with Cached Jacobians
class EnhancedHybridURT(AdaptiveURT):
    """Hybrid URT with cached Jacobians and optimized mode switching"""

    def __init__(self, alpha: float = 1.155, theta_h: float = 2.4,
                 beta_min: float = 0.235, beta_max: float = 0.5,
                 mode_switch_threshold: int = 100, state_dim: int = 100,
                 cache_jacobians: bool = True):
        super().__init__(alpha, theta_h, beta_min, beta_max, state_dim)

        self.mode_switch_threshold = mode_switch_threshold
        self.current_mode = 'adaptive'
        self.mode_history = []
        self.local_optimizers = {}
        self.jacobian_cache = {}
        self.cache_jacobians = cache_jacobians
        self.performance_comparison = []

    def step(self, P: np.ndarray, u_input: float = 0.05) -> np.ndarray:
        """Enhanced hybrid step with performance monitoring"""
        n = len(P)
        previous_mode = self.current_mode

        # Enhanced mode switching with hysteresis
        if n < 10:
            new_mode = 'lqr'
        elif n < 30:
            new_mode = 'mpc_fast'
        elif n < 80:
            new_mode = 'mpc'
        elif n < 150:
            new_mode = 'adaptive_aggressive'
        else:
            new_mode = 'adaptive'

        # Mode transition logic with performance consideration
        if new_mode != self.current_mode:
            if len(self.performance_comparison) > 5:
                recent_perf = np.mean([p['performance'] for p in self.performance_comparison[-5:]])
                if recent_perf > 0.9:  # Good performance, be cautious about switching
                    if 'adaptive' in self.current_mode and n < 200:
                        new_mode = self.current_mode  # Maintain current mode

        self.current_mode = new_mode

        # Execute appropriate control strategy
        if self.current_mode == 'lqr':
            P_next = self.lqr_step(P, u_input)
        elif self.current_mode == 'mpc_fast':
            P_next = self.mpc_step(P, u_input, horizon=3, max_iter=10)
        elif self.current_mode == 'mpc':
            P_next = self.mpc_step(P, u_input, horizon=5, max_iter=20)
        elif self.current_mode == 'adaptive_aggressive':
            P_next = self.aggressive_adaptive_step(P, u_input)
        else:
            P_next = super().step(P, u_input)

        # Performance tracking
        performance = self.assess_step_performance(P, P_next, previous_mode)
        self.performance_comparison.append(performance)
        self.mode_history.append({
            'step': len(self.mode_history),
            'mode': self.current_mode,
            'performance': performance,
            'dimension': n
        })

        return P_next

    def aggressive_adaptive_step(self, P: np.ndarray, u_input: float) -> np.ndarray:
        """Aggressive adaptive control for medium-sized systems"""
        # Use larger beta range for faster convergence
        aggressive_beta_max = min(self.beta_max * 1.3, 0.65)
        temp_beta_max = self.beta_max
        self.beta_max = aggressive_beta_max

        try:
            P_next = super().step(P, u_input)
        finally:
            self.beta_max = temp_beta_max  # Restore original

        return P_next

    def lqr_step(self, P: np.ndarray, u_input: float) -> np.ndarray:
        """Enhanced LQR controller with cached gains"""
        n = len(P)
        key = f"lqr_{n}"

        if key not in self.local_optimizers:
            # Compute LQR gain with cached Jacobian
            A = self.compute_or_cache_jacobian(P, key)
            B = np.eye(n)
            Q = np.eye(n)
            R = 0.1 * np.eye(n)

            # Solve discrete-time algebraic Riccati equation
            P_riccati = self.solve_dare(A, B, Q, R)
            K = np.linalg.inv(R + B.T @ P_riccati @ B) @ B.T @ P_riccati @ A
            self.local_optimizers[key] = K

        K = self.local_optimizers[key]
        return K @ P + u_input * np.ones_like(P)

    def compute_or_cache_jacobian(self, P: np.ndarray, key: str) -> np.ndarray:
        """Compute Jacobian with caching support"""
        if self.cache_jacobians and key in self.jacobian_cache:
            return self.jacobian_cache[key]

        J = self.compute_jacobian(P)

        if self.cache_jacobians:
            self.jacobian_cache[key] = J

        return J

    def compute_jacobian(self, P: np.ndarray) -> np.ndarray:
        """Compute Jacobian of URT dynamics"""
        n = len(P)
        J = np.zeros((n, n))
        h = 1e-6

        for i in range(n):
            P_plus = P.copy()
            P_minus = P.copy()
            P_plus[i] += h
            P_minus[i] -= h

            # Finite difference approximation
            f_plus = self.urt_dynamics(P_plus, 0.0)
            f_minus = self.urt_dynamics(P_minus, 0.0)
            J[:, i] = (f_plus - f_minus) / (2 * h)

        return J

    def urt_dynamics(self, P: np.ndarray, u_input: float) -> np.ndarray:
        """URT dynamics for MPC prediction"""
        phi_P = self.phi(P)
        nonlinear_term = self.alpha * (P - self.theta_h * phi_P)
        return self.beta * (nonlinear_term + u_input * np.ones_like(P))

    def solve_dare(self, A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray, max_iter: int = 100) -> np.ndarray:
        """Solve Discrete-time Algebraic Riccati Equation"""
        P = Q.copy()
        for _ in range(max_iter):
            P_next = A.T @ P @ A - A.T @ P @ B @ np.linalg.inv(R + B.T @ P @ B) @ B.T @ P @ A + Q
            if np.max(np.abs(P_next - P)) < 1e-6:
                break
            P = P_next
        return P

    def mpc_step(self, P: np.ndarray, u_input: float, horizon: int = 5,
                 max_iter: int = 20) -> np.ndarray:
        """Enhanced MPC with warm starting and caching"""
        n = len(P)

        def mpc_cost(U_flat: np.ndarray) -> float:
            """MPC cost function with stability penalty"""
            U_seq = U_flat.reshape(horizon, n)
            total_cost = 0.0
            P_pred = P.copy()

            for k in range(horizon):
                # Predict next state using URT dynamics
                P_pred = self.urt_dynamics(P_pred, U_seq[k])
                # Quadratic cost with terminal cost
                state_cost = P_pred.T @ P_pred
                control_cost = 0.1 * U_seq[k].T @ U_seq[k]

                # Terminal cost for stability
                if k == horizon - 1:
                    state_cost *= 2.0

                total_cost += state_cost + control_cost

            return total_cost

        # Warm start from previous solution if available
        U0 = np.zeros(horizon * n)
        warm_start_key = f"mpc_warm_{n}_{horizon}"
        if warm_start_key in self.local_optimizers:
            U0 = self.local_optimizers[warm_start_key]

        # Optimize control sequence
        bounds = [(-1.0, 1.0) for _ in range(horizon * n)]

        result = minimize(mpc_cost, U0, method='L-BFGS-B',
                         bounds=bounds, options={'maxiter': max_iter})

        # Cache warm start for next iteration
        self.local_optimizers[warm_start_key] = result.x

        # Apply first control input
        U_opt = result.x.reshape(horizon, n)
        return self.urt_dynamics(P, U_opt[0])

    def assess_step_performance(self, P_prev: np.ndarray, P_current: np.ndarray,
                              previous_mode: str) -> float:
        """Assess step performance for mode switching decisions"""
        error_reduction = np.linalg.norm(P_prev) - np.linalg.norm(P_current)
        relative_reduction = error_reduction / (np.linalg.norm(P_prev) + 1e-12)

        # Normalize to [0, 1] range
        performance = np.clip(relative_reduction * 10, 0, 1)
        return performance

# Cell 7: Run Enhanced Hybrid URT Demo
print("\n" + "="*50)
print("Enhanced Hybrid URT Demo")
print("="*50)

hybrid_urt = EnhancedHybridURT(state_dim=50)
hybrid_sim = PlasmaSimulator(state_dim=50, urt_controller=hybrid_urt)
hybrid_trajectory = hybrid_sim.simulate_plasma_with_control(steps=150)
hybrid_sim.plot_results()

hybrid_suppression = np.mean(hybrid_sim.mode_suppression[-50:]) * 100
print(f"\nHybrid URT Performance:")
print(f"Average Final Mode Suppression: {hybrid_suppression:.1f}%")
print(f"Mode History: {[m['mode'] for m in hybrid_urt.mode_history[-10:]]}")

# Cell 8: Performance Comparison Visualization
print("\n" + "="*50)
print("Performance Comparison")
print("="*50)

# Compare all three controllers
controllers = {
    'Base URT': base_sim,
    'Adaptive URT': plasma_sim,
    'Hybrid URT': hybrid_sim
}

plt.figure(figsize=(15, 5))

# Plot 1: State norm comparison
plt.subplot(1, 3, 1)
for name, sim in controllers.items():
    plt.plot(sim.time_steps, sim.state_norms, label=name, linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('State Norm')
plt.title('Convergence Comparison')
plt.legend()
plt.grid(True)

# Plot 2: Mode suppression comparison
plt.subplot(1, 3, 2)
for name, sim in controllers.items():
    plt.plot(sim.time_steps, sim.mode_suppression, label=name, linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Mode Suppression')
plt.title('Stabilization Performance')
plt.legend()
plt.grid(True)

# Plot 3: Control effort comparison
plt.subplot(1, 3, 3)
control_efforts = []
names = []
for name, sim in controllers.items():
    control_efforts.append(np.mean(np.abs(sim.control_inputs)))
    names.append(name)
plt.bar(names, control_efforts, alpha=0.7)
plt.ylabel('Mean Control Effort')
plt.title('Control Efficiency')
plt.grid(True)

plt.tight_layout()
plt.show()

# Final summary
print("\n=== FINAL SUMMARY ===")
for name, sim in controllers.items():
    final_suppression = np.mean(sim.mode_suppression[-50:]) * 100
    convergence_steps = next((i for i, n in enumerate(sim.state_norms) if n < 0.1), len(sim.state_norms))
    print(f"{name:15} | Suppression: {final_suppression:5.1f}% | Convergence: {convergence_steps:3d} steps | Control: {np.mean(np.abs(sim.control_inputs)):.4f}")

print(f"\nBest overall: {max(controllers.items(), key=lambda x: np.mean(x[1].mode_suppression[-50:]))[0]}")

Initializing Plasma Simulator with Adaptive URT...


ValueError: System unstable: κ=1.443 >= 1